In [ ]:
## 서울, 

In [ ]:
import time
import pandas as pd
import os
import certifi
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

# SSL 인증서 경로 설정
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

def crawl_ediya_seoul():
    # 1. 서울시 구 리스트
    seoul_gu_list = [
        "강남구", "강동구", "강북구", "강서구", "관악구", "광진구", "구로구", "금천구", 
        "노원구", "도봉구", "동대문구", "동작구", "마포구", "서대문구", "서초구", "성동구", 
        "성북구", "송파구", "양천구", "영등포구", "용산구", "은평구", "종로구", "중구", "중랑구"
    ]

    options = webdriver.ChromeOptions()
    # options.add_argument('--headless') 
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    wait = WebDriverWait(driver, 10)
    
    # URL 수정 (기존 URL에서 변경됨 확인 필요)
    url = "https://members.ediya.com/store#none"
    driver.get(url)
    
    all_stores = []

    try:
        # 2. '주소' 탭 클릭
        address_tab = wait.until(EC.element_to_be_clickable((By.XPATH, "//a[contains(@onclick, 'storeAddr')]")))
        address_tab.click()
        time.sleep(1)

        for gu in seoul_gu_list:
            start_time = time.time()
            
            try:
                # 3. 주소 입력칸에 구 이름 입력
                search_input = wait.until(EC.presence_of_element_located((By.ID, "keyword")))
                search_input.clear()
                search_input.send_keys(f"{gu}") 
                
                # 4. 검색 버튼 클릭
                search_btn = driver.find_element(By.CLASS_NAME, "btn_search")
                driver.execute_script("arguments[0].click();", search_btn)
                
                # 5. 결과 로딩 대기
                time.sleep(2) 
                
                # 6. BeautifulSoup 파싱
                html = driver.page_source
                soup = BeautifulSoup(html, 'html.parser')
                items = soup.select('.store_list .st_li')
                
                gu_store_count = 0
                for item in items:
                    name = item.select_one('.name').get_text(strip=True)
                    addr = item.select_one('.addr').get_text(strip=True)
                    
                    all_stores.append({
                        '매장명': name,
                        '주소': addr,
                        '구': gu
                    })
                    gu_store_count += 1
                
                end_time = time.time()
                print(f"[{gu}] 완료! - 매장 수: {gu_store_count:2d}개 | 소요시간: {end_time - start_time:.2f}초")

            except Exception as e:
                print(f"{gu} 크롤링 중 에러 발생: {e}")

    finally:
        # 10. 최종 데이터 정제 및 저장
        if all_stores:
            df = pd.DataFrame(all_stores)
            
            # [정제 1] 중복 데이터 제거
            df = df.drop_duplicates(subset=['매장명', '주소'])
            
            # [정제 2] 주소가 '서울'로 시작하지 않는 데이터 삭제
            # 유저분께서 말씀하신 "매장명이 서울로 안 시작하는" 부분을 "주소가 서울인 것만 남기기"로 적용했습니다.
            initial_count = len(df)
            df = df[df['주소'].str.startswith('서울')]
            filtered_count = initial_count - len(df)
            
            df.to_csv("ediya.csv", index=False, encoding="utf-8-sig")
            
            print("\n" + "="*50)
            print(f"최종 정제 완료!")
            print(f"- 중복 제거 후 데이터: {initial_count}건")
            print(f"- 타 지역(서울 외) 제거: {filtered_count}건")
            print(f"- 최종 저장 데이터: {len(df)}건")
            print("="*50)
        
        driver.quit()

if __name__ == "__main__":
    crawl_ediya_seoul()

[강남구] 완료! - 매장 수: 62개 | 소요시간: 2.98초
[강동구] 완료! - 매장 수: 38개 | 소요시간: 2.85초
[강북구] 완료! - 매장 수: 18개 | 소요시간: 2.63초
[강서구] 완료! - 매장 수: 50개 | 소요시간: 2.78초
[관악구] 완료! - 매장 수: 32개 | 소요시간: 2.83초
[광진구] 완료! - 매장 수: 30개 | 소요시간: 2.62초
[구로구] 완료! - 매장 수: 32개 | 소요시간: 2.59초
[금천구] 완료! - 매장 수: 28개 | 소요시간: 2.58초
[노원구] 완료! - 매장 수: 34개 | 소요시간: 2.59초
[도봉구] 완료! - 매장 수: 28개 | 소요시간: 2.68초
[동대문구] 완료! - 매장 수: 36개 | 소요시간: 2.67초
[동작구] 완료! - 매장 수: 36개 | 소요시간: 2.55초
[마포구] 완료! - 매장 수: 34개 | 소요시간: 2.77초
[서대문구] 완료! - 매장 수: 30개 | 소요시간: 2.57초
[서초구] 완료! - 매장 수: 52개 | 소요시간: 2.92초
[성동구] 완료! - 매장 수: 28개 | 소요시간: 2.57초
[성북구] 완료! - 매장 수: 38개 | 소요시간: 2.54초
[송파구] 완료! - 매장 수: 38개 | 소요시간: 2.57초
[양천구] 완료! - 매장 수: 20개 | 소요시간: 2.54초
[영등포구] 완료! - 매장 수: 68개 | 소요시간: 2.77초
[용산구] 완료! - 매장 수: 30개 | 소요시간: 2.62초
[은평구] 완료! - 매장 수: 40개 | 소요시간: 2.56초
[종로구] 완료! - 매장 수: 56개 | 소요시간: 2.60초
[중구] 완료! - 매장 수: 100개 | 소요시간: 2.91초
[중랑구] 완료! - 매장 수: 48개 | 소요시간: 2.69초

최종 정제 완료!
- 중복 제거 후 데이터: 503건
- 타 지역(서울 외) 제거: 38건
- 최종 저장 데이터: 465건


In [25]:
import time
import pandas as pd
import os
import certifi
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

# SSL 인증서 경로 설정
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

def crawl_ediya_jeju():
    # 1. 제주특별자치도 시 리스트
    jeju_list = ["제주시", "서귀포시"]

    target_file = "ediya.csv"
    
    # [중복 방지] 기존 파일 로드 (서울부터 광주까지의 모든 데이터 기억)
    seen_stores = set()
    if os.path.exists(target_file):
        try:
            existing_df = pd.read_csv(target_file)
            for _, row in existing_df.iterrows():
                seen_stores.add((row['매장명'], row['주소']))
            print(f"기존 파일에서 {len(seen_stores)}개의 지점을 확인했습니다. 마지막 제주도 수집을 시작합니다.")
        except Exception as e:
            print(f"기존 파일 읽기 실패: {e}")

    options = webdriver.ChromeOptions()
    # options.add_argument('--headless') 
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    wait = WebDriverWait(driver, 10)
    
    # 요청하신 URL
    url = "https://members.ediya.com/store#none"
    driver.get(url)
    
    new_collected_data = []

    try:
        # 2. '주소' 탭 클릭
        address_tab = wait.until(EC.element_to_be_clickable((By.XPATH, "//a[contains(@onclick, 'storeAddr')]")))
        address_tab.click()
        time.sleep(1)

        for area in jeju_list:
            start_time = time.time()
            current_area_count = 0
            
            try:
                # 3. 주소 입력칸에 {area}만 입력 (제주시, 서귀포시)
                search_input = wait.until(EC.presence_of_element_located((By.ID, "keyword")))
                search_input.clear()
                search_input.send_keys(area) 
                
                # 4. 검색 버튼 클릭
                search_btn = driver.find_element(By.CLASS_NAME, "btn_search")
                driver.execute_script("arguments[0].click();", search_btn)
                
                # 5. 결과 로딩 대기
                time.sleep(2.5) 
                
                # 6. BeautifulSoup 파싱
                html = driver.page_source
                soup = BeautifulSoup(html, 'html.parser')
                items = soup.select('.store_list .st_li')
                
                for item in items:
                    name_tag = item.select_one('.name')
                    addr_tag = item.select_one('.addr')
                    
                    if name_tag and addr_tag:
                        name = name_tag.get_text(strip=True)
                        addr = addr_tag.get_text(strip=True)
                        
                        # [중복 체크] 이미 수집된 지점인지 확인
                        if (name, addr) not in seen_stores:
                            new_collected_data.append({
                                '매장명': name,
                                '주소': addr,
                                '지역': area
                            })
                            seen_stores.add((name, addr))
                            current_area_count += 1
                
                end_time = time.time()
                print(f"[{area}] 완료 - 신규 수집: {current_area_count:2d}개 | 소요시간: {end_time - start_time:.2f}초")

            except Exception as e:
                print(f"{area} 크롤링 중 오류: {e}")

    finally:
        # 모든 수집이 끝난 후 최종 정제 및 저장
        if new_collected_data:
            new_df = pd.DataFrame(new_collected_data)
            
            # [최종 정제] 주소에 '제주'가 포함된 데이터만 필터링
            initial_count = len(new_df)
            new_df = new_df[new_df['주소'].str.contains('제주')]
            filtered_count = initial_count - len(new_df)
            
            if os.path.exists(target_file):
                new_df.to_csv(target_file, index=False, encoding="utf-8-sig", mode='a', header=False)
                print(f"\n기존 {target_file}에 제주 데이터 {len(new_df)}건을 추가했습니다.")
            else:
                new_df.to_csv(target_file, index=False, encoding="utf-8-sig")
                print(f"\n새로운 {target_file} 파일을 생성하고 저장했습니다.")
            
            if filtered_count > 0:
                print(f"(정제 알림: 주소지에 '제주'가 포함되지 않은 {filtered_count}건의 데이터가 제외되었습니다.)")
        else:
            print("\n새로 추가할 데이터가 없습니다.")
        
        # 전체 데이터 개수 최종 확인
        if os.path.exists(target_file):
            final_df = pd.read_csv(target_file)
            print(f"\n🎉 축하합니다! 전국 이디야 매장 수집이 완료되었습니다.")
            print(f"📍 최종 저장된 매장 수: {len(final_df)}개")
        
        driver.quit()

if __name__ == "__main__":
    crawl_ediya_jeju()

기존 파일에서 2221개의 지점을 확인했습니다. 마지막 제주도 수집을 시작합니다.
[제주시] 완료 - 신규 수집: 12개 | 소요시간: 3.53초
[서귀포시] 완료 - 신규 수집:  7개 | 소요시간: 3.30초

기존 ediya.csv에 제주 데이터 19건을 추가했습니다.

🎉 축하합니다! 전국 이디야 매장 수집이 완료되었습니다.
📍 최종 저장된 매장 수: 2240개
